<a href="https://colab.research.google.com/github/JawadSk12/ML-Lab/blob/main/ML_COMPETITION_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e3_path = kagglehub.competition_download('playground-series-s6e3')

print('Data source import complete.')


In [ ]:
# ==========================================
# CUSTOMER CHURN PREDICTION - FULL SCRIPT
# Playground Series S6E3
# ==========================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier


# ==========================================
# CONFIG
# ==========================================
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
SUBMISSION_FILE = "submission.csv"

TARGET = "Churn"
ID_COL = "id"
RANDOM_STATE = 42


# ==========================================
# LOAD DATA
# ==========================================
def load_data():
    print("Loading data...")
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    return train, test


# ==========================================
# PREPROCESSING
# ==========================================
def preprocess(train, test):
    print("Preprocessing data...")

    # Separate target
    y = train[TARGET].map({"No": 0, "Yes": 1})
    X = train.drop([TARGET, ID_COL], axis=1)

    test_ids = test[ID_COL]
    test = test.drop(ID_COL, axis=1)

    # Identify categorical columns
    cat_cols = X.select_dtypes(include=["object"]).columns

    # Label Encoding
    le_dict = {}
    for col in cat_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        test[col] = le.transform(test[col])
        le_dict[col] = le

    # Feature Scaling
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    test = scaler.transform(test)

    return X, y, test, test_ids


# ==========================================
# TRAIN MODEL
# ==========================================
def train_model(X, y):
    print("Training model...")

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        random_state=RANDOM_STATE,
        use_label_encoder=False
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )

    # Validation score
    y_pred_val = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred_val)

    print(f"\nValidation AUC Score: {auc:.5f}")

    return model


# ==========================================
# PREDICT + SAVE SUBMISSION
# ==========================================
def create_submission(model, test, test_ids):
    print("Generating predictions...")

    preds = model.predict_proba(test)[:, 1]

    submission = pd.DataFrame({
        ID_COL: test_ids,
        TARGET: preds
    })

    submission.to_csv(SUBMISSION_FILE, index=False)

    print(f"Submission saved as {SUBMISSION_FILE}")


# ==========================================
# MAIN PIPELINE
# ==========================================
def main():
    train, test = load_data()

    X, y, test_processed, test_ids = preprocess(train, test)

    model = train_model(X, y)

    create_submission(model, test_processed, test_ids)


# ==========================================
# RUN
# ==========================================
if __name__ == "__main__":
    main()

Loading data...
Preprocessing data...
Training model...
[0]	validation_0-auc:0.90276
[100]	validation_0-auc:0.91370
[200]	validation_0-auc:0.91494
[300]	validation_0-auc:0.91588
[400]	validation_0-auc:0.91640
[500]	validation_0-auc:0.91671
[600]	validation_0-auc:0.91690
[700]	validation_0-auc:0.91699
[800]	validation_0-auc:0.91706
[900]	validation_0-auc:0.91710
[999]	validation_0-auc:0.91712

Validation AUC Score: 0.91712
Generating predictions...
Submission saved as submission.csv
